<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Métricas de Distancia y Estandarización
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/01_Metricas_de_Distancia_y_Estandarizacion.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. ¿Por qué Importan las Métricas de Distancia? 🧭

Todo algoritmo de clustering necesita, en su núcleo, una forma de cuantificar **qué tan "parecidas" o "diferentes"** son dos observaciones. Esta cuantificación se conoce como **métrica de distancia** (o, de forma más general, medida de disimilitud).

La elección de la métrica **no es un detalle técnico menor**: cambia por completo lo que el algoritmo entiende por "similar", y una métrica mal elegida para la escala o la naturaleza de los datos puede producir agrupamientos engañosos. En este cuaderno estudiaremos cuatro métricas fundamentales — **Euclidiana, Manhattan, Coseno y Hamming** — y veremos por qué, en la mayoría de los casos, es indispensable **estandarizar** las variables antes de calcular cualquier distancia.

---
## Configuración del Entorno de Trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path

    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

# Cargamos de una vez el dataset que usaremos a lo largo de este cuaderno
file_mall = load_dataset('mall_customers.csv')
df_mall = pd.read_csv(file_mall)

print("🚀 Entorno configurado exitosamente para el Módulo 10: Métricas de Distancia.")
print(f"📦 Dataset 'mall_customers.csv' cargado: {df_mall.shape[0]} filas x {df_mall.shape[1]} columnas.")

---
## 2. Distancia Euclidiana (Norma $L_2$) 📏

La **distancia Euclidiana** es la distancia en línea recta entre dos puntos — la noción de distancia más intuitiva, heredada de la geometría clásica. Es sensible a la escala de las variables.

Para dos puntos $p=(p_1, p_2)$ y $q=(q_1, q_2)$ en el plano:

$$d_{2}(p, q) = \sqrt{(p_1 - q_1)^2 + (p_2 - q_2)^2}$$

De forma general, para $p, q \in \mathbb{R}^n$:

$$d_{2}(p, q) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

También se le conoce como **norma $L_2$**.

In [ ]:
from scipy.spatial.distance import euclidean
from sklearn.metrics import pairwise_distances

p = np.array([2, 3])
q = np.array([8, 7])

d_manual = np.sqrt(np.sum((p - q) ** 2))
d_scipy = euclidean(p, q)

print(f"Distancia Euclidiana (cálculo manual): {d_manual:.4f}")
print(f"Distancia Euclidiana (scipy.spatial.distance): {d_scipy:.4f}")

---
## 3. Distancia Manhattan (Norma $L_1$) 🚕

La **distancia Manhattan** (también llamada *cityblock distance* o **norma $L_1$**) suma las diferencias absolutas de cada coordenada, en lugar de la raíz de la suma de cuadrados. También es sensible a la escala, pero **menos sensible a valores atípicos (outliers)** que la distancia Euclidiana, ya que no eleva las diferencias al cuadrado.

$$d_{1}(p, q) = \sum_{i=1}^{n} |p_i - q_i|$$

<div align="center">
  <img src="images/manhattan_distance.svg" width="380" alt="Distancia Manhattan" style="border-radius:8px; box-shadow: 0 4px 6px rgba(0,0,0,0.12); margin: 10px 0; background:white; padding:8px;"/>
  <p style="font-size: 0.85em; color: #64748b;">
    <i>Figura: La distancia Manhattan recorre una cuadrícula (como un taxi en Manhattan), mientras que la Euclidiana va en línea recta ("como vuela el cuervo").</i>
  </p>
</div>

In [ ]:
from scipy.spatial.distance import cityblock

d_manhattan_manual = np.sum(np.abs(p - q))
d_manhattan_scipy = cityblock(p, q)

print(f"Distancia Manhattan (cálculo manual): {d_manhattan_manual}")
print(f"Distancia Manhattan (scipy.spatial.distance): {d_manhattan_scipy}")
print(f"Recordatorio -> Distancia Euclidiana entre los mismos puntos: {d_manual:.4f}")

---
## 4. Distancia Coseno 📐

La **distancia coseno** no mide la distancia en línea recta, sino el **ángulo** entre dos vectores. Se define a partir de la **similitud coseno**, que es el coseno del ángulo $\theta$ entre los vectores:

$$\text{sim}_{\cos}(p, q) = \cos(\theta) = \frac{p \cdot q}{\|p\| \, \|q\|}$$

$$d_{\cos}(p, q) = 1 - \text{sim}_{\cos}(p, q)$$

A diferencia de la Euclidiana o la Manhattan, la distancia coseno es **invariante a la magnitud** de los vectores — solo le importa su *dirección*, no su tamaño. Esto la hace especialmente popular en **clustering de texto** (por ejemplo, vectores TF-IDF), donde dos documentos pueden tratar del mismo tema con longitudes muy distintas.

In [ ]:
from scipy.spatial.distance import cosine

vector_a = np.array([4, 0, 1])          # p.ej. frecuencias de palabras en el documento A
vector_b = np.array([2, 0, 0.5])        # documento B: mismas proporciones, la mitad de longitud

d_cos = cosine(vector_a, vector_b)
d_euclid_vectores = euclidean(vector_a, vector_b)

print(f"Distancia Coseno entre A y B:     {d_cos:.4f}")
print(f"Distancia Euclidiana entre A y B: {d_euclid_vectores:.4f}")

Observe cómo la distancia coseno se aproxima a 0 (indicando que ambos vectores apuntan prácticamente en la misma dirección), mientras que la distancia Euclidiana es considerablemente mayor pese a tratarse de vectores "proporcionalmente idénticos" — solo escalados en magnitud. Esto ilustra por qué la distancia coseno es preferible cuando lo que importa es la *proporción* entre componentes y no su tamaño absoluto.

---
## 5. Distancia de Hamming (Métrica Adicional) 🔤

La **distancia de Hamming** aplica a secuencias discretas de igual longitud (cadenas de texto, cadenas binarias, secuencias categóricas) y cuenta simplemente **en cuántas posiciones difieren** dos secuencias.

$$d_{H}(p, q) = \sum_{i=1}^{n} \mathbb{1}[p_i \neq q_i], \qquad \text{con } |p| = |q| = n$$

**Ejemplos:**
* `"CASA"` vs. `"CASO"` → difieren solo en la última posición → $d_H = 1$.
* `"10101"` vs. `"01101"` → difieren en la 1ª y 2ª posición → $d_H = 2$.

⚠️ **Limitación importante:** la distancia de Hamming **solo está definida para secuencias de igual longitud** y no permite inserciones ni eliminaciones — únicamente sustituciones posición por posición. Por ejemplo, no es posible calcular directamente $d_H(\text{"MARIA"}, \text{"MARA"})$ porque las cadenas tienen 5 y 4 caracteres respectivamente: para compararlas habría que rellenar o truncar artificialmente una de ellas, lo cual distorsiona el resultado.

Cuando se necesita comparar cadenas de **longitud distinta** permitiendo inserciones, eliminaciones y sustituciones, la métrica adecuada es la **distancia de Levenshtein (*edit distance*)**, que cuenta el número mínimo de operaciones de edición necesarias para transformar una cadena en otra.

In [ ]:
def hamming_distance(s1: str, s2: str) -> int:
    if len(s1) != len(s2):
        raise ValueError("La distancia de Hamming requiere cadenas de igual longitud.")
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

print(f"Hamming('CASA', 'CASO')    = {hamming_distance('CASA', 'CASO')}")
print(f"Hamming('10101', '01101') = {hamming_distance('10101', '01101')}")

### Bonus: Distancia de Levenshtein (*Edit Distance*) — cuando las longitudes difieren

In [ ]:
def levenshtein_distance(s1: str, s2: str) -> int:
    n, m = len(s1), len(s2)
    dp = np.zeros((n + 1, m + 1), dtype=int)
    dp[:, 0] = np.arange(n + 1)
    dp[0, :] = np.arange(m + 1)

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            costo_sustitucion = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,                     # eliminación
                dp[i, j - 1] + 1,                     # inserción
                dp[i - 1, j - 1] + costo_sustitucion  # sustitución
            )
    return int(dp[n, m])

print(f"Levenshtein('MARIA', 'MARA') = {levenshtein_distance('MARIA', 'MARA')}")

### Tabla Comparativa de Métricas de Distancia

| Métrica | Fórmula | Sensible a Escala | Sensible a Outliers | Caso de Uso Típico |
|---|---|---|---|---|
| **Euclidiana ($L_2$)** | $\sqrt{\sum (p_i-q_i)^2}$ | Sí | Alta (eleva al cuadrado) | K-Means, datos continuos de escala comparable |
| **Manhattan ($L_1$)** | $\sum \lvert p_i-q_i \rvert$ | Sí | Menor que Euclidiana | Datos con outliers, alta dimensionalidad |
| **Coseno** | $1 - \cos(\theta)$ | No (invariante a magnitud) | Baja | Texto (TF-IDF), sistemas de recomendación |
| **Hamming** | $\sum \mathbb{1}[p_i \neq q_i]$ | No aplica (categórica) | No aplica | Cadenas/secuencias categóricas de igual longitud |

---
## 6. La Importancia de la Estandarización antes de Agrupar ⚖️

Estandarizar los datos antes de aplicar clustering **no siempre es obligatorio**, pero su importancia depende directamente de la métrica de distancia elegida. Con la **distancia Euclidiana** en particular, las variables con rangos numéricos más amplios **dominan** el cálculo de la distancia y terminan teniendo una influencia desproporcionada sobre el agrupamiento — aunque no sean realmente "más importantes", simplemente están medidas en unidades más grandes.

Este mismo problema afecta a otros algoritmos basados en distancia o varianza, como **k-NN** y **PCA**, por la misma razón. La solución estándar es usar `sklearn.preprocessing.StandardScaler`, que transforma cada variable para que tenga **media 0 y varianza 1**.

El dataset `mall_customers.csv` es un ejemplo perfecto: `Age` está en un rango completamente distinto al de `Annual_Income_k` (ingresos en miles). Sin escalar, `Annual_Income_k` dominaría cualquier clustering basado en distancia Euclidiana solo por tener números más grandes en bruto — aunque `Age` podría ser igual de relevante para agrupar a los clientes.

In [ ]:
df_mall[['Age', 'Annual_Income_k', 'Spending_Score']].describe()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

features = ['Age', 'Annual_Income_k']
X_raw = df_mall[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Distancias par-a-par entre las primeras 5 observaciones, sin escalar vs. escaladas
dist_raw = euclidean_distances(X_raw[:5])
dist_scaled = euclidean_distances(X_scaled[:5])

print("Distancias Euclidianas SIN escalar (Annual_Income_k domina por su rango):")
print(np.round(dist_raw, 2))
print()
print("Distancias Euclidianas ESCALADAS (Age y Annual_Income_k pesan de forma comparable):")
print(np.round(dist_scaled, 2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].boxplot([df_mall['Age'], df_mall['Annual_Income_k']], labels=['Age', 'Annual_Income_k'])
axes[0].set_title("Antes de Escalar: Rangos Muy Distintos")

axes[1].boxplot([X_scaled[:, 0], X_scaled[:, 1]], labels=['Age', 'Annual_Income_k'])
axes[1].set_title("Después de StandardScaler: Rangos Comparables")

plt.tight_layout()
plt.show()

---
##### 🛠️ Práctica 1: Distancias Antes y Después de Escalar en `mall_customers.csv`

**Objetivo:** Comprobar de forma práctica cómo la estandarización puede **cambiar cuál es el vecino más cercano** de un cliente, usando las variables `Age` y `Annual_Income_k` de `mall_customers.csv`.

**Instrucciones:**
1. Selecciona un cliente objetivo, por ejemplo el de `CustomerID == 1` (usa `df_mall`, ya cargado en la celda de configuración del entorno).
2. Usando `sklearn.metrics.pairwise.euclidean_distances`, calcula la distancia del cliente objetivo a **todos los demás clientes**, con las columnas `Age` y `Annual_Income_k` **sin escalar**. Identifica el `CustomerID` del vecino más cercano.
3. Repite el cálculo, pero esta vez con las columnas **estandarizadas** con `StandardScaler`. Identifica de nuevo el vecino más cercano.
4. Compara: ¿el vecino más cercano es el mismo en ambos casos? Explica en una celda markdown por qué escalar (o no) puede cambiar la respuesta.

In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 1 - Vecino Más Cercano Antes y Después de Escalar
# =========================================================================

# 1. Seleccionar cliente objetivo y variables
# target_idx = df_mall.index[df_mall['CustomerID'] == 1][0]
# features = ['Age', 'Annual_Income_k']
# X_raw = df_mall[features].values

# 2. Distancias SIN escalar y vecino más cercano
# dist_raw = euclidean_distances(X_raw)[target_idx]
# ...

# 3. Escalar con StandardScaler y recalcular distancias
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_raw)
# dist_scaled = euclidean_distances(X_scaled)[target_idx]
# ...

# 4. Comparar los dos vecinos más cercanos encontrados

<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
features = ['Age', 'Annual_Income_k']
X_raw = df_mall[features].values

target_idx = df_mall.index[df_mall['CustomerID'] == 1][0]

# --- Sin escalar ---
dist_raw = euclidean_distances(X_raw)[target_idx]
dist_raw[target_idx] = np.inf  # ignorar la distancia del cliente consigo mismo
vecino_raw = df_mall.iloc[np.argmin(dist_raw)]['CustomerID']

# --- Escalado ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
dist_scaled = euclidean_distances(X_scaled)[target_idx]
dist_scaled[target_idx] = np.inf
vecino_scaled = df_mall.iloc[np.argmin(dist_scaled)]['CustomerID']

print(f"Vecino más cercano (SIN escalar):  CustomerID = {vecino_raw}")
print(f"Vecino más cercano (ESCALADO):     CustomerID = {vecino_scaled}")
print(f"¿Es el mismo vecino en ambos casos? {vecino_raw == vecino_scaled}")
```

**Interpretación esperada:** Es común que el vecino más cercano **cambie** entre ambos escenarios. Sin escalar, `Annual_Income_k` (con un rango numérico mucho mayor que `Age`) domina el cálculo de la distancia, por lo que el "vecino más cercano" tiende a ser simplemente el cliente con el ingreso más parecido, casi sin importar la edad. Al estandarizar, ambas variables contribuyen de forma comparable, y la noción de "cercanía" refleja mejor una similitud conjunta en edad **e** ingreso.
</details>

---
### 7. Resumen y Conclusiones del Cuaderno 01 📌

1. **La Métrica Define la Similitud:** Euclidiana, Manhattan, Coseno y Hamming capturan nociones distintas de "parecido" — la elección correcta depende de la naturaleza de los datos (continuos, categóricos, texto) y del problema.
2. **Escalar No Es Opcional con Distancia Euclidiana:** Sin `StandardScaler`, las variables con rangos numéricos más grandes dominan artificialmente el cálculo de distancias y, por tanto, el resultado del clustering.
3. **Próximos Pasos:** Con las métricas de distancia y la estandarización dominadas, en el siguiente cuaderno aplicaremos el algoritmo de clustering más popular — **K-Means** — sobre datos ya correctamente preparados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>